# **Imports**

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import fetch_covtype
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import numpy as np
import copy

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Το μοντέλο θα τρέξει σε: {device}")

Το μοντέλο θα τρέξει σε: cuda


# **Data Preprocessing**

In [3]:
print("\nΦόρτωση Forest Covertype Dataset (Μπορεί να πάρει 1-2 λεπτά λόγω μεγέθους)...")
data = fetch_covtype()

X = data.data
y = data.target - 1

print(f"Συνολικές εγγραφές: {X.shape[0]}, Χαρακτηριστικά: {X.shape[1]}")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

num_tasks = 5
patients_per_task = len(X_scaled) // num_tasks

train_loaders = []
test_loaders = []
BATCH_SIZE = 512

for i in range(num_tasks):
    start_idx = i * patients_per_task
    end_idx = (i + 1) * patients_per_task if i < num_tasks - 1 else len(X_scaled)

    X_task = X_scaled[start_idx:end_idx]
    y_task = y[start_idx:end_idx]

    X_train, X_test, y_train, y_test = train_test_split(X_task, y_task, test_size=0.1, random_state=42)

    train_dataset = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.long))
    test_dataset = TensorDataset(torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test, dtype=torch.long))

    train_loaders.append(DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True))
    test_loaders.append(DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False))

print(f"Δημιουργήθηκαν {num_tasks} Tasks. Κάθε Task περιέχει δεδομένα από ~{patients_per_task} περιοχές.")


Φόρτωση Forest Covertype Dataset (Μπορεί να πάρει 1-2 λεπτά λόγω μεγέθους)...
Συνολικές εγγραφές: 581012, Χαρακτηριστικά: 54
Δημιουργήθηκαν 5 Tasks. Κάθε Task περιέχει δεδομένα από ~116202 περιοχές.


# **Models**

In [4]:
INPUT_DIM = X.shape[1]
LATENT_DIM = 16
HIDDEN_DIM = 128
NUM_CLASSES = 7

class TabularVAE(nn.Module):
    def __init__(self):
        super(TabularVAE, self).__init__()
        self.fc1 = nn.Linear(INPUT_DIM, HIDDEN_DIM)
        self.fc2_mu = nn.Linear(HIDDEN_DIM, LATENT_DIM)
        self.fc2_logvar = nn.Linear(HIDDEN_DIM, LATENT_DIM)

        self.fc3 = nn.Linear(LATENT_DIM, HIDDEN_DIM)
        self.fc4 = nn.Linear(HIDDEN_DIM, INPUT_DIM)

    def encode(self, x):
        h1 = F.relu(self.fc1(x))
        return self.fc2_mu(h1), self.fc2_logvar(h1)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h3 = F.relu(self.fc3(z))
        return self.fc4(h3)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon_x = self.decode(z)
        return recon_x, mu, logvar

class MLPClassifier(nn.Module):
    def __init__(self):
        super(MLPClassifier, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(INPUT_DIM, HIDDEN_DIM),
            nn.ReLU(),
            nn.BatchNorm1d(HIDDEN_DIM),
            nn.Dropout(0.3),
            nn.Linear(HIDDEN_DIM, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Linear(64, NUM_CLASSES)
        )

    def forward(self, x):
        return self.net(x)

def vae_loss_function(recon_x, x, mu, logvar):
    MSE = F.mse_loss(recon_x, x, reduction='sum')
    KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return MSE + KLD

def evaluate_tabular(model, test_loaders, current_task_idx):
    model.eval()
    accuracies = []
    with torch.no_grad():
        for i in range(current_task_idx + 1):
            correct = 0
            total = 0
            for data, target in test_loaders[i]:
                data, target = data.to(device), target.to(device)
                outputs = model(data)
                _, predicted = torch.max(outputs.data, 1)
                total += target.size(0)
                correct += (predicted == target).sum().item()
            accuracies.append(100 * correct / total)
    return accuracies

# **Trainig with Pure Generative Replay**

In [5]:
classifier = MLPClassifier().to(device)
vae = TabularVAE().to(device)

opt_classifier = optim.Adam(classifier.parameters(), lr=0.001)
opt_vae = optim.Adam(vae.parameters(), lr=0.001)

prev_classifier = None
prev_vae = None
EPOCHS = 10

print("\n--- Ξεκινάει η Εκπαίδευση (Massive Scale: Covtype Dataset) ---")

for task_idx, train_loader in enumerate(train_loaders):
    print(f"\nTask {task_idx + 1}: Εκπαίδευση σε νέο chunk δεδομένων...")

    for epoch in range(EPOCHS):
        classifier.train()
        vae.train()

        for real_data, real_target in train_loader:
            real_data, real_target = real_data.to(device), real_target.to(device)
            batch_size = real_data.size(0)

            if prev_vae is not None and prev_classifier is not None:
                with torch.no_grad():
                    fake_batch_size = batch_size * task_idx
                    z = torch.randn(fake_batch_size, LATENT_DIM).to(device)
                    fake_data = prev_vae.decode(z)

                    fake_outputs = prev_classifier(fake_data)
                    _, fake_target = torch.max(fake_outputs.data, 1)

                combined_data = torch.cat([real_data, fake_data])
                combined_target = torch.cat([real_target, fake_target])
            else:
                combined_data = real_data
                combined_target = real_target

            opt_classifier.zero_grad()
            outputs = classifier(combined_data)
            loss_class = F.cross_entropy(outputs, combined_target)
            loss_class.backward()
            opt_classifier.step()

            opt_vae.zero_grad()
            recon_batch, mu, logvar = vae(combined_data)
            loss_vae = vae_loss_function(recon_batch, combined_data, mu, logvar)
            loss_vae.backward()
            opt_vae.step()

    accs = evaluate_tabular(classifier, test_loaders, task_idx)
    print(f"Ακρίβεια Ταξινόμησης μετά το Task {task_idx + 1}:")
    for i, acc in enumerate(accs):
        print(f" -> Δεδομένα Task {i + 1}: {acc:.2f}%")

    prev_classifier = copy.deepcopy(classifier)
    prev_classifier.eval()
    prev_vae = copy.deepcopy(vae)
    prev_vae.eval()


--- Ξεκινάει η Εκπαίδευση (Massive Scale: Covtype Dataset) ---

Task 1: Εκπαίδευση σε νέο chunk δεδομένων...
Ακρίβεια Ταξινόμησης μετά το Task 1:
 -> Δεδομένα Task 1: 89.72%

Task 2: Εκπαίδευση σε νέο chunk δεδομένων...
Ακρίβεια Ταξινόμησης μετά το Task 2:
 -> Δεδομένα Task 1: 83.20%
 -> Δεδομένα Task 2: 88.26%

Task 3: Εκπαίδευση σε νέο chunk δεδομένων...
Ακρίβεια Ταξινόμησης μετά το Task 3:
 -> Δεδομένα Task 1: 80.29%
 -> Δεδομένα Task 2: 74.91%
 -> Δεδομένα Task 3: 86.81%

Task 4: Εκπαίδευση σε νέο chunk δεδομένων...
Ακρίβεια Ταξινόμησης μετά το Task 4:
 -> Δεδομένα Task 1: 77.12%
 -> Δεδομένα Task 2: 70.19%
 -> Δεδομένα Task 3: 69.06%
 -> Δεδομένα Task 4: 89.84%

Task 5: Εκπαίδευση σε νέο chunk δεδομένων...
Ακρίβεια Ταξινόμησης μετά το Task 5:
 -> Δεδομένα Task 1: 78.06%
 -> Δεδομένα Task 2: 70.36%
 -> Δεδομένα Task 3: 61.87%
 -> Δεδομένα Task 4: 68.01%
 -> Δεδομένα Task 5: 91.00%
